# 대한민국 산업단지·초중학교 위치 지도

산업단지와 운영 중인 초등학교·중학교 위치를 표시합니다.\n산업단지는 빨간색, 초등학교는 파란색, 중학교는 보라색 점으로 구분합니다.\n바탕지도 없이 대한민국 외곽선과 시·도 경계만 표시하며, 지도는 elem_map.html로 저장됩니다.

In [ ]:
from pathlib import Path
from html import escape
import json
import math
import struct
import folium
import pandas as pd
from IPython.display import display

# 프로젝트 루트나 zoning 폴더 어느 쪽에서 실행해도 자료를 찾습니다.
search_roots = [Path.cwd(), Path.cwd() / 'zoning', Path.cwd().parent]
shp_path = next((candidate for root in search_roots
                 for candidate in (root / 'indcluster data').glob('*/dam_pdan.shp')), None)
if shp_path is None:
    raise FileNotFoundError('indcluster data/*/dam_pdan.shp 파일을 찾지 못했습니다.')
dbf_path = shp_path.with_suffix('.dbf')
school_path = next((candidate for root in search_roots
                    for candidate in (root / 'indcluster data').glob('*.csv')), None)
if school_path is None:
    raise FileNotFoundError('indcluster data 안의 초중등학교 CSV 파일을 찾지 못했습니다.')
boundary_path = Path(__file__).with_name('skorea-provinces-2018-topo-simple.json') if '__file__' in globals() else None
if boundary_path is None or not boundary_path.exists():
    candidates = [Path.cwd() / 'skorea-provinces-2018-topo-simple.json',
                  Path.cwd() / 'zoning' / 'skorea-provinces-2018-topo-simple.json',
                  Path.cwd().parent / 'skorea-provinces-2018-topo-simple.json']
    boundary_path = next((p for p in candidates if p.exists()), None)
if boundary_path is None:
    raise FileNotFoundError('시도 경계 파일 skorea-provinces-2018-topo-simple.json을 찾지 못했습니다.')

def read_dbf(path):
    data = path.read_bytes()
    count = int.from_bytes(data[4:8], 'little')
    header_len = int.from_bytes(data[8:10], 'little')
    record_len = int.from_bytes(data[10:12], 'little')
    fields = []
    for offset in range(32, header_len - 1, 32):
        desc = data[offset:offset + 32]
        name = desc[:11].split(b'\x00', 1)[0].decode('ascii')
        if name:
            fields.append((name, desc[16]))
    records = []
    for index in range(count):
        start = header_len + index * record_len
        raw = data[start:start + record_len]
        if not raw or raw[0:1] == b'*':
            records.append(None)
            continue
        cursor, attrs = 1, {}
        for name, length in fields:
            value = raw[cursor:cursor + length].strip()
            cursor += length
            try:
                text = value.decode('utf-8').strip()
            except UnicodeDecodeError:
                text = value.decode('cp949', errors='replace').strip()
            attrs[name] = text
        records.append(attrs)
    return records

def read_shapefile_points(path):
    data = path.read_bytes()
    shape_type = struct.unpack_from('<i', data, 32)[0]
    if shape_type != 1:
        raise ValueError(f'Point Shapefile이 아닙니다 (shape type={shape_type}).')
    points, offset = [], 100
    while offset + 8 <= len(data):
        size = struct.unpack_from('>i', data, offset + 4)[0] * 2
        start = offset + 8
        record_type = struct.unpack_from('<i', data, start)[0]
        points.append(struct.unpack_from('<dd', data, start + 4) if record_type == 1 else None)
        offset = start + size
    return points

def to_lonlat(x, y):
    radius = 6378137.0
    return (math.degrees(x / radius),
            math.degrees(2 * math.atan(math.exp(y / radius)) - math.pi / 2))

# TopoJSON의 공유 경계 arc를 GeoJSON 좌표로 복원합니다.
topology = json.loads(boundary_path.read_text(encoding='utf-8'))
scale = topology['transform']['scale']
translate = topology['transform']['translate']
decoded_arcs = []
for arc in topology['arcs']:
    x = y = 0
    coordinates = []
    for dx, dy in arc:
        x += dx
        y += dy
        coordinates.append([x * scale[0] + translate[0], y * scale[1] + translate[1]])
    decoded_arcs.append(coordinates)

def join_arcs(indices):
    coordinates = []
    for arc_index in indices:
        arc = decoded_arcs[arc_index if arc_index >= 0 else ~arc_index]
        if arc_index < 0:
            arc = list(reversed(arc))
        coordinates.extend(arc if not coordinates else arc[1:])
    return coordinates

def polygon_coordinates(rings):
    return [join_arcs(ring) for ring in rings]

topology_object = next(iter(topology['objects'].values()))
boundary_features = []
arc_use_count = {}
for geometry in topology_object['geometries']:
    kind = geometry['type']
    arcs = geometry['arcs']
    if kind == 'Polygon':
        coordinates = polygon_coordinates(arcs)
        rings = arcs
    elif kind == 'MultiPolygon':
        coordinates = [polygon_coordinates(polygon) for polygon in arcs]
        rings = [ring for polygon in arcs for ring in polygon]
    else:
        continue
    for ring in rings:
        for arc_index in ring:
            key = arc_index if arc_index >= 0 else ~arc_index
            arc_use_count[key] = arc_use_count.get(key, 0) + 1
    boundary_features.append({'type': 'Feature', 'properties': geometry.get('properties', {}),
                              'geometry': {'type': kind, 'coordinates': coordinates}})
province_geojson = {'type': 'FeatureCollection', 'features': boundary_features}
# 한 시도에만 속하는 arc는 대한민국의 바깥 경계입니다.
national_outline = {'type': 'MultiLineString',
                    'coordinates': [decoded_arcs[i] for i, count in arc_use_count.items() if count == 1]}

attributes = read_dbf(dbf_path)
points = read_shapefile_points(shp_path)
if len(attributes) != len(points):
    raise ValueError(f'SHP/DBF 레코드 수 불일치: {len(points)} / {len(attributes)}')
field_labels = {'DAN_ID': '산업단지 ID', 'DAN_NAME': '산업단지명',
                'DANJI_TYPE': '산업단지 유형', 'CRE_DATE': '작성일',
                'UPD_DATE': '수정일', 'GOSI': '고시', 'REMARK': '비고'}
features = []
for point, attrs in zip(points, attributes):
    if point is None or attrs is None:
        continue
    lon, lat = to_lonlat(*point)
    if -180 <= lon <= 180 and -90 <= lat <= 90:
        features.append((lat, lon, attrs))
print(f'표시할 산업단지: {len(features):,}개')
print(f'시도 경계: {len(boundary_features)}개')

school_df = pd.read_csv(school_path, encoding='utf-8-sig')
required = ['학교명', '학교급구분', '운영상태', '소재지도로명주소', '위도', '경도']
missing = [column for column in required if column not in school_df.columns]
if missing:
    raise ValueError(f'초중등학교 CSV에 필요한 열이 없습니다: {missing}')
active_schools = school_df['운영상태'].astype('string').str.strip().eq('운영')
school_level = school_df['학교급구분'].astype('string').str.strip()
elementary_schools = school_df.loc[active_schools & school_level.eq('초등학교')].copy()
middle_schools = school_df.loc[active_schools & school_level.eq('중학교')].copy()

def clean_school_coordinates(frame):
    frame = frame.copy()
    frame['위도'] = pd.to_numeric(frame['위도'], errors='coerce')
    frame['경도'] = pd.to_numeric(frame['경도'], errors='coerce')
    frame = frame.dropna(subset=['위도', '경도'])
    return frame[frame['위도'].between(33, 39) & frame['경도'].between(124, 132)]

elementary_schools = clean_school_coordinates(elementary_schools)
middle_schools = clean_school_coordinates(middle_schools)
print(f'표시할 초등학교: {len(elementary_schools):,}개')
print(f'표시할 중학교: {len(middle_schools):,}개')


In [ ]:
# 배경 타일을 사용하지 않아 도로·지명 등 다른 지도 정보는 나타나지 않습니다.
indcluster_map = folium.Map(location=[36.3, 127.8], zoom_start=7,
                           tiles=None, control_scale=False, prefer_canvas=True)
indcluster_map.get_root().html.add_child(folium.Element(
    '<style>html,body{background:#e8eef2!important}.leaflet-container{background:#e8eef2}</style>'
))
folium.GeoJson(
    province_geojson,
    name='시·도 경계',
    style_function=lambda feature: {'color': '#65727a', 'weight': 1.2,
                                    'fillColor': '#ffffff', 'fillOpacity': 1},
).add_to(indcluster_map)
folium.GeoJson(
    {'type': 'Feature', 'properties': {}, 'geometry': national_outline},
    name='대한민국 경계',
    style_function=lambda feature: {'color': '#26343b', 'weight': 2.8,
                                    'opacity': 1, 'fillOpacity': 0},
).add_to(indcluster_map)
# 경계 폴리곤 다음에 점 레이어를 추가해 점들이 경계 채움색에 가려지지 않게 합니다.
industry_layer = folium.FeatureGroup(name='산업단지', show=True).add_to(indcluster_map)
school_layers = [
    ('초등학교', elementary_schools, '#2196f3'),
    ('중학교', middle_schools, '#8e44ad'),
]
school_marker_layers = {}
for level, schools, color in school_layers:
    layer = folium.FeatureGroup(name=level, show=True).add_to(indcluster_map)
    school_marker_layers[level] = layer

for lat, lon, attrs in features:
    rows = ''.join(
        f'<tr><th style="text-align:left;padding:3px 8px">{escape(field_labels.get(k, k))}</th>'
        f'<td style="padding:3px 8px">{escape(v)}</td></tr>'
        for k, v in attrs.items() if v
    )
    name = attrs.get('DAN_NAME') or '산업단지'
    popup = folium.Popup(f'<div style="max-height:300px;overflow:auto"><b>{escape(name)}</b>'
                         f'<table>{rows}</table></div>', max_width=420)
    folium.CircleMarker(location=[lat, lon], radius=3, color='#b63e32', weight=0.6,
                        fill=True, fill_color='#d84b3f', fill_opacity=0.9,
                        tooltip=name, popup=popup).add_to(industry_layer)
folium.LayerControl(collapsed=False).add_to(indcluster_map)

for level, schools, color in school_layers:
    layer = school_marker_layers[level]
    for name, address, lat, lon in zip(
        schools['학교명'].fillna('초중학교'),
        schools['소재지도로명주소'].fillna(''),
        schools['위도'],
        schools['경도'],
    ):
        name, address = str(name), str(address)
        popup = folium.Popup(f'<b>{escape(name)}</b><br>{escape(address)}', max_width=360)
        folium.CircleMarker(
            location=[lat, lon], radius=2.2, color=color, weight=0.5,
            fill=True, fill_color=color, fill_opacity=0.9,
            tooltip=name, popup=popup,
        ).add_to(layer)
all_lons = [coord[0] for feature in boundary_features
            for polygon in (feature['geometry']['coordinates'] if feature['geometry']['type'] == 'MultiPolygon'
                           else [feature['geometry']['coordinates']])
            for ring in polygon for coord in ring]
all_lats = [coord[1] for feature in boundary_features
            for polygon in (feature['geometry']['coordinates'] if feature['geometry']['type'] == 'MultiPolygon'
                           else [feature['geometry']['coordinates']])
            for ring in polygon for coord in ring]
indcluster_map.fit_bounds([[min(all_lats), min(all_lons)], [max(all_lats), max(all_lons)]], padding=(18, 18))
output_path = Path.cwd() / 'elem_map.html'
if not (output_path.parent / 'indcluster data').exists():
    output_path = shp_path.parent.parent.parent / 'elem_map.html'
indcluster_map.save(output_path)
print(f'지도 저장 완료: {output_path.resolve()}')
display(indcluster_map)